# 03 — Results Analysis

Compare all trained baselines and generate final visualisations:
- Side-by-side metrics table (AUC-ROC per class)
- ROC curves for the most prevalent pathologies
- t-SNE of pre-trained vs. fine-tuned embeddings
- Loss curves

**Prerequisite:** Run evaluation for each mode:
```bash
python evaluate.py --checkpoint checkpoints/finetune/best_model_full_finetune.pth --mode full_finetune
python evaluate.py --checkpoint checkpoints/finetune/best_model_linear_probe.pth --mode linear_probe
python evaluate.py --checkpoint checkpoints/finetune/best_model_imagenet_baseline.pth --mode imagenet_baseline
```

In [ ]:
import os
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.dataset import ALL_CLASSES

LOGS_DIR = '../logs'
print('Libraries loaded.')

## 1. Load metrics from all evaluation runs

In [ ]:
def load_metrics(metrics_path):
    """Parse a metrics .txt file into a dict {class_name: auc_roc}."""
    if not os.path.exists(metrics_path):
        return None
    metrics = {}
    with open(metrics_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 4 and parts[0] in ALL_CLASSES:
                metrics[parts[0]] = float(parts[1])  # AUC-ROC
            elif 'Macro AUC-ROC' in line:
                metrics['Macro AUC-ROC'] = float(line.split(':')[1].strip())
    return metrics


modes = ['full_finetune', 'linear_probe', 'imagenet_baseline']
mode_labels = {'full_finetune': 'SimCLR + Full Finetune',
               'linear_probe': 'SimCLR + Linear Probe',
               'imagenet_baseline': 'ImageNet Baseline'}

all_metrics = {}
for mode in modes:
    path = os.path.join(LOGS_DIR, f'metrics_{mode}.txt')
    m = load_metrics(path)
    if m:
        all_metrics[mode_labels[mode]] = m
        print(f'Loaded: {mode} — Macro AUC-ROC: {m.get("Macro AUC-ROC", "N/A")}')
    else:
        print(f'Not found: {path}')

## 2. Comparison table

In [ ]:
if all_metrics:
    rows = []
    for cls_name in ALL_CLASSES:
        row = {'Class': cls_name}
        for label, metrics in all_metrics.items():
            row[label] = metrics.get(cls_name, float('nan'))
        rows.append(row)
    
    # Add macro average row
    macro_row = {'Class': 'MACRO AVG'}
    for label, metrics in all_metrics.items():
        macro_row[label] = metrics.get('Macro AUC-ROC', float('nan'))
    rows.append(macro_row)
    
    results_df = pd.DataFrame(rows).set_index('Class')
    
    # Highlight best per row
    styled = results_df.style.highlight_max(axis=1, props='font-weight: bold; color: green;').format('{:.4f}')
    print('AUC-ROC per class (higher is better):')
    display(styled)
else:
    print('No metrics files found. Run evaluation first.')

## 3. Loss curves

In [ ]:
from src.evaluation.visualize import plot_loss_curves
plot_loss_curves(log_dir=LOGS_DIR, save_path=os.path.join(LOGS_DIR, 'loss_curves.png'))

img = plt.imread(os.path.join(LOGS_DIR, 'loss_curves.png'))
plt.figure(figsize=(14, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

## 4. t-SNE embeddings

In [ ]:
tsne_path = os.path.join(LOGS_DIR, 'tsne.png')
if os.path.exists(tsne_path):
    img = plt.imread(tsne_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('t-SNE of Fine-tuned Encoder Embeddings')
    plt.show()
else:
    print(f't-SNE plot not found at {tsne_path}. Run: python evaluate.py ...')

## 5. ROC curves

In [ ]:
roc_path = os.path.join(LOGS_DIR, 'roc_curves.png')
if os.path.exists(roc_path):
    img = plt.imread(roc_path)
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print(f'ROC curves not found at {roc_path}. Run evaluate.py first.')

## 6. GradCAM visualisation

In [ ]:
gcam_path = os.path.join(LOGS_DIR, 'gradcam_pneumonia.png')
if os.path.exists(gcam_path):
    img = plt.imread(gcam_path)
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('GradCAM — Pneumonia Class Activation Maps')
    plt.show()
else:
    print(f'GradCAM not found. Run: python evaluate.py --checkpoint <path>')